In [ ]:
%matplotlib notebook
from rfsoc_rfdc.rfsoc_overlay import RFSoCOverlay
from rfsoc_rfdc.overlay_task import OverlayTask
from rfsoc_rfdc.overlay_task import BlinkLedTask

from rfsoc_rfdc.transmitter.single_ch_tx_task import SingleChTxTask
from rfsoc_rfdc.receiver.single_ch_rx_task import SingleChRxTask

from rfsoc_rfdc.transmitter.multi_ch_tx_indept_task import MultiChTxIndeptTask
from rfsoc_rfdc.receiver.multi_ch_rx_indept_task import MultiChRxIndeptTask

from rfsoc_rfdc.rfdc_task import RfdcTask 
from rfsoc_rfdc.cmac_task import CmacTask

from rfsoc_rfdc.rfdc_config import ZCU216_CONFIG

import sys
import os
import time

In [ ]:
from rfsoc_rfdc.dsp.ofdm import OFDM
from rfsoc_rfdc.dsp.detection import Detection

In [ ]:
ol = RFSoCOverlay(path_to_bitstream="./rfsoc_rfdc/bitstream/rfsoc_rfdc_v45_rt_2ch_mts.bit")
NEW_CONFIG = {
    "RefClockForPLL": 300.0,
    "DACSampleRate": 2400.0,
    "DACInterpolationRate": 4,
    "DACNCO": 800,
    "ADCSampleRate": 2400.0,
    "ADCInterpolationRate": 4,
    "ADCNCO": -800                                           
}
ZCU216_CONFIG.update(NEW_CONFIG)

In [ ]:
rfdc_t = RfdcTask(ol, mts_enable=True, debug_mode=True, board="ZCU216")

for task in [rfdc_t]:
    task.start()
    task.join()

In [ ]:
true_samp_rate = ZCU216_CONFIG['DACSampleRate'] / ZCU216_CONFIG['DACInterpolationRate'] * 1e6

for qam_order in ["QPSK", "16QAM", "64QAM", "256QAM"]:

    for atten in range(0, 40, 3):

        ZCU216_CONFIG['OFDM_ATTEN_DB'] = atten
        ZCU216_CONFIG['QAM'] = qam_order
        ZCU216_CONFIG['OFDM_SCHEME'] = OFDM(sym_num=100, fft_size=64, sub_num=48, modu=qam_order, cp_rate=0.25)
        ZCU216_CONFIG['DETECTION_SCHEME'] = Detection(sample_rate=true_samp_rate)

        print(f"QAM={qam_order}, OFDM_ATTEN_DB={atten}")

        led_t = BlinkLedTask(ol)
        tx_t = MultiChTxIndeptTask(ol, mode="iq2real", channel_count=2, dp_vect_dim=2)
        rx_t = MultiChRxIndeptTask(ol, mode="real2iq", channel_count=2, dp_vect_dim=2)

        parallel_task = [led_t, tx_t, rx_t]
        for task in parallel_task:
            task.start()

        time.sleep(30)

        for task in parallel_task:
            task.stop()